# 06 Advanced Multi-Representation Domain-Robust SER

Notebook này bắt đầu thử nghiệm kiến trúc advanced cho bài toán **Speech Emotion Recognition (SER)** trong hướng **Presentation Feedback System**.

Mục tiêu không chỉ là tăng accuracy trên `combined_random`, mà còn tạo kiến trúc có khả năng tổng quát hóa tốt hơn khi gặp speaker/dataset mới.

## Kiến trúc tổng quan

```text
Input audio 16 kHz
      |
      |-- Branch A: Temporal acoustic branch
      |      MFCC + delta + delta-delta + RMS + ZCR + spectral features
      |      -> 1D-CNN blocks
      |      -> BiLSTM
      |      -> attention pooling
      |      -> z_temporal
      |
      |-- Branch B: Spectrogram branch
      |      log-Mel + delta log-Mel + delta-delta log-Mel
      |      -> 2D-CNN / small ResNet
      |      -> SE/channel attention
      |      -> z_spectral
      |
      |-- Branch C: Pretrained speech branch
      |      raw waveform
      |      -> frozen WavLM-base-plus
      |      -> adapter MLP
      |      -> z_wavlm
      |
      |-- Branch D: Statistical branch
             handcrafted statistical vector
             -> stats MLP embedding for fusion
             -> RBF-SVM branch for probability-level ensemble

Fusion:
      z_fused = concat(z_temporal, z_spectral, z_wavlm, z_stats)
      |-- emotion head -> 6 emotion probabilities
      |-- optional domain-adversarial head -> reduce dataset leakage

Final:
      stacking ensemble over [p_deep_fusion, p_rbf_svm]
```

## Điểm mới so với notebook 05

- Thêm **2D log-Mel CNN + SE attention**, vì kết quả 05 cho thấy thiếu nhánh spectral mạnh để kéo `combined_random`.
- Giữ **BiLSTM temporal branch** để học diễn tiến cảm xúc theo thời gian.
- Giữ **WavLM frozen embedding** vì nhánh WavLM trong 05 đang mạnh nhất.
- Giữ **RBF-SVM trên rich statistical features** vì nhẹ, dễ giải thích và hỗ trợ báo cáo.
- Thêm tùy chọn **Domain-Adversarial Training** để giảm overfit theo dataset/speaker signature.


## 1. Paper References and Why They Are Used

| Kỹ thuật | Paper / nguồn | Lý do dùng trong notebook |
|---|---|---|
| Domain-adversarial training + Gradient Reversal Layer | Ganin et al., *Domain-Adversarial Training of Neural Networks* ([arXiv:1505.07818](https://arxiv.org/abs/1505.07818)) | Giảm dấu vết dataset/speaker trong fused embedding |
| WavLM frozen speech representation | Chen et al., *WavLM: Large-Scale Self-Supervised Pre-training for Full Stack Speech Processing* ([arXiv:2110.13900](https://arxiv.org/abs/2110.13900)) | Lấy representation speech mạnh từ raw waveform |
| Squeeze-and-Excitation | Hu et al., *Squeeze-and-Excitation Networks* ([arXiv:1709.01507](https://arxiv.org/abs/1709.01507)) | Reweight channel feature map trong 2D-CNN |
| SpecAugment | Park et al., *SpecAugment* ([arXiv:1904.08779](https://arxiv.org/abs/1904.08779)) | Mask time/frequency trên spectrogram để giảm overfit |
| mixup | Zhang et al., *mixup: Beyond Empirical Risk Minimization* ([arXiv:1710.09412](https://arxiv.org/abs/1710.09412)) | Làm mềm decision boundary, giảm memorization |
| Ensemble CNN-LSTM-GRU SER | Ahmed et al., *An Ensemble 1D-CNN-LSTM-GRU Model with Data Augmentation for SER* ([arXiv:2112.05666](https://arxiv.org/abs/2112.05666)) | Gợi ý dùng recurrent CNN variants + augmentation + ensemble |
| Feature engineering + BiLSTM/DeepCRF SER | Chowdhury et al., *A Novel Hybrid Deep Learning Technique for Speech Emotion Detection using Feature Engineering* ([arXiv:2507.07046](https://arxiv.org/abs/2507.07046)) | Gợi ý rich handcrafted features + recurrent modeling |

Các PDF nền tảng đã được tải vào:

```text
Papers for main models emotion classification/papers/
```


## 2. Công Thức Chính

### 2.1 Emotion classification

Với fused embedding \(z\), emotion head tạo logits:

\[
\hat{y} = \mathrm{softmax}(W_e z + b_e)
\]

Loss chính:

\[
\mathcal{L}_{emo} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)
\]

Trong notebook dùng `CrossEntropyLoss` với optional label smoothing.

### 2.2 Attention pooling

BiLSTM tạo chuỗi hidden states:

\[
H = [h_1, h_2, ..., h_T]
\]

Attention score:

\[
e_t = v^\top \tanh(W h_t)
\]

Attention weight:

\[
\alpha_t = \frac{\exp(e_t)}{\sum_{k=1}^{T}\exp(e_k)}
\]

Context vector:

\[
z = \sum_{t=1}^{T}\alpha_t h_t
\]

Ý nghĩa: model tự chọn frame quan trọng nhất cho cảm xúc.

### 2.3 Squeeze-and-Excitation

Với feature map \(U \in \mathbb{R}^{C \times H \times W}\):

\[
s_c = \frac{1}{HW}\sum_i\sum_j U_{c,i,j}
\]

\[
a = \sigma(W_2 \delta(W_1 s))
\]

\[
\tilde{U}_c = a_c U_c
\]

Ý nghĩa: 2D-CNN học channel nào quan trọng hơn trên log-Mel spectrogram.

### 2.4 Gradient Reversal Layer

Forward:

\[
\mathrm{GRL}(z) = z
\]

Backward:

\[
\frac{\partial \mathrm{GRL}}{\partial z} = -\lambda I
\]

Domain head cố đoán dataset:

\[
\hat{d} = \mathrm{softmax}(W_d \mathrm{GRL}(z) + b_d)
\]

Total loss:

\[
\mathcal{L} = \mathcal{L}_{emo} + \beta \mathcal{L}_{domain}
\]

Nhờ GRL, encoder/fusion bị ép làm representation khó đoán domain hơn, trong khi vẫn giữ thông tin emotion.

### 2.5 mixup

Với hai sample \((x_i, y_i)\), \((x_j, y_j)\):

\[
\tilde{x} = \lambda x_i + (1-\lambda)x_j
\]

\[
\tilde{y} = \lambda y_i + (1-\lambda)y_j
\]

\[
\lambda \sim \mathrm{Beta}(\alpha, \alpha)
\]

Ý nghĩa: giảm overfit bằng cách làm ranh giới giữa các lớp mềm hơn.


## 3. Cảnh Báo Về Adversarial

Domain-adversarial training có thể giúp giảm overfit theo dataset/speaker, nhưng không phải lúc nào cũng tăng accuracy.

Trong SER, speaker cues và emotion cues có thể chồng lên nhau:

- Pitch vừa liên quan giới tính/speaker, vừa liên quan emotion.
- Energy vừa là thói quen nói, vừa là tín hiệu angry/excited.
- Timbre là speaker trait, nhưng emotion cũng làm đổi timbre.

Vì vậy notebook mặc định:

```text
USE_DOMAIN_ADVERSARIAL=1
ADV_LAMBDA_MAX=0.05
ADV_WARMUP_EPOCHS=5
```

Nghĩa là adversarial chỉ bật nhẹ sau warm-up. Khi viết báo cáo, phải có ablation:

```text
without adversarial
with domain adversarial lambda=0.02/0.05/0.1
```


In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU = os.getenv("USE_MULTI_GPU", "1") == "1"
torch.backends.cudnn.benchmark = False

def maybe_data_parallel(model, name="model"):
    if USE_MULTI_GPU and GPU_COUNT > 1:
        print(f"Using DataParallel for {name} on {GPU_COUNT} GPUs")
        return nn.DataParallel(model)
    return model

print("Device:", DEVICE)
print("GPU_COUNT:", GPU_COUNT, "USE_MULTI_GPU:", USE_MULTI_GPU)


## 4. Configuration

Kaggle quick test:

```text
QUICK_RUN=1
RUN_PRETRAINED_SPEECH=0
MAX_EPOCHS=1
RUN_SINGLE_DATASET=0
```

Kaggle full test:

```text
QUICK_RUN=0
RUN_PRETRAINED_SPEECH=1
PRETRAINED_SPEECH_MODEL=microsoft/wavlm-base-plus
MAX_EPOCHS=35
USE_DOMAIN_ADVERSARIAL=1
ADV_LAMBDA_MAX=0.05
USE_MULTI_GPU=1
```


In [ ]:
QUICK_RUN = os.getenv("QUICK_RUN", "0") == "1"
QUICK_RUN_PER_GROUP = int(os.getenv("QUICK_RUN_PER_GROUP", "18"))

TARGET_SR = 16_000
TARGET_DURATION = float(os.getenv("TARGET_DURATION", "3.0"))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)
N_FFT = int(os.getenv("N_FFT", "400"))
WIN_LENGTH = int(os.getenv("WIN_LENGTH", "400"))
HOP_LENGTH = int(os.getenv("HOP_LENGTH", "160"))
N_MFCC = int(os.getenv("N_MFCC", "40"))
N_MELS = int(os.getenv("N_MELS", "96"))
MAX_FRAMES = int(1 + TARGET_LENGTH // HOP_LENGTH)

COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]
LABEL_TO_ID = {label: i for i, label in enumerate(COMMON_EMOTIONS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "48"))
MAX_EPOCHS = int(os.getenv("MAX_EPOCHS", "35"))
PATIENCE = int(os.getenv("PATIENCE", "8"))
LR = float(os.getenv("LR", "1e-3"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "3e-4"))
DROPOUT = float(os.getenv("DROPOUT", "0.35"))
LABEL_SMOOTHING = float(os.getenv("LABEL_SMOOTHING", "0.06"))

RUN_COMBINED_RANDOM = os.getenv("RUN_COMBINED_RANDOM", "1") == "1"
RUN_COMBINED_STRICT = os.getenv("RUN_COMBINED_STRICT", "1") == "1"
RUN_SINGLE_DATASET = os.getenv("RUN_SINGLE_DATASET", "1") == "1"

RUN_PRETRAINED_SPEECH = os.getenv("RUN_PRETRAINED_SPEECH", "1") == "1"
PRETRAINED_SPEECH_MODEL = os.getenv("PRETRAINED_SPEECH_MODEL", "microsoft/wavlm-base-plus")
SPEECH_EMBED_BATCH_SIZE = int(os.getenv("SPEECH_EMBED_BATCH_SIZE", "16" if GPU_COUNT > 1 else "8"))

USE_AUGMENTATION = os.getenv("USE_AUGMENTATION", "1") == "1"
USE_WAVEFORM_AUGMENTATION = os.getenv("USE_WAVEFORM_AUGMENTATION", "1") == "1"
WAVEFORM_AUGMENT_COPIES = int(os.getenv("WAVEFORM_AUGMENT_COPIES", "1"))
TEMPORAL_WAVE_AUG_PROB = float(os.getenv("TEMPORAL_WAVE_AUG_PROB", "0.45"))
SPECTRAL_AUG_PROB = float(os.getenv("SPECTRAL_AUG_PROB", "0.45"))
MIXUP_ALPHA = float(os.getenv("MIXUP_ALPHA", "0.10"))
MIXUP_PROB = float(os.getenv("MIXUP_PROB", "0.20"))

USE_CLASS_WEIGHTS = os.getenv("USE_CLASS_WEIGHTS", "1") == "1"
USE_BALANCED_SAMPLER = os.getenv("USE_BALANCED_SAMPLER", "1") == "1"
CACHE_FEATURES = os.getenv("CACHE_FEATURES", "1") == "1"
ENABLE_RICH_PITCH = os.getenv("ENABLE_RICH_PITCH", "1") == "1"
STATS_PCA_COMPONENTS = int(os.getenv("STATS_PCA_COMPONENTS", "256"))

USE_DOMAIN_ADVERSARIAL = os.getenv("USE_DOMAIN_ADVERSARIAL", "1") == "1"
ADV_LAMBDA_MAX = float(os.getenv("ADV_LAMBDA_MAX", "0.05"))
ADV_WARMUP_EPOCHS = int(os.getenv("ADV_WARMUP_EPOCHS", "5"))
DOMAIN_LOSS_WEIGHT = float(os.getenv("DOMAIN_LOSS_WEIGHT", "1.0"))

print({
    "QUICK_RUN": QUICK_RUN,
    "MAX_EPOCHS": MAX_EPOCHS,
    "RUN_PRETRAINED_SPEECH": RUN_PRETRAINED_SPEECH,
    "PRETRAINED_SPEECH_MODEL": PRETRAINED_SPEECH_MODEL,
    "USE_DOMAIN_ADVERSARIAL": USE_DOMAIN_ADVERSARIAL,
    "ADV_LAMBDA_MAX": ADV_LAMBDA_MAX,
    "USE_WAVEFORM_AUGMENTATION": USE_WAVEFORM_AUGMENTATION,
})


In [ ]:
def find_ser_processed():
    candidates = []
    env_path = os.getenv("SER_PROCESSED", "").strip()
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        Path("/kaggle/input/datasets/quanghuy225/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed"),
        Path("/kaggle/working/ser_processed"),
        Path.cwd() / "ser_processed",
        Path.cwd().parent / "ser_processed",
        Path.cwd().parent / "01&02_Data_and_DataProcessing" / "ser_processed",
        Path("D:/UTE/Speech Programming/Speech Project/01&02_Data_and_DataProcessing/ser_processed"),
    ])
    for candidate in candidates:
        if (candidate / "metadata.csv").exists():
            return candidate.resolve()
    for root in [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for metadata_path in root.rglob("metadata.csv"):
                if metadata_path.parent.name == "ser_processed":
                    return metadata_path.parent.resolve()
    raise FileNotFoundError("Cannot find ser_processed/metadata.csv")

SER_PROCESSED = find_ser_processed()
AUDIO_16K_DIR = SER_PROCESSED / "audio_16k"
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "06_Advanced_Model_outputs"
REPORT_DIR = OUTPUT_DIR / "reports"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
CACHE_DIR = OUTPUT_DIR / "cache"
for d in [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("SER_PROCESSED:", SER_PROCESSED)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 5. Metadata and Protocols

Notebook giữ ba nhóm protocol:

- `combined_random`: gần với nhiều paper/repo dùng random split.
- `combined_strict_no_tess`: speaker-aware hơn, loại TESS vì chỉ có 2 speaker.
- `single-dataset`: train/test riêng từng corpus để so với paper thường báo theo từng dataset.

Điểm quan trọng: augmentation chỉ được dùng trong train loader hoặc train index, validation/test luôn dùng feature gốc.


In [ ]:
metadata = pd.read_csv(SER_PROCESSED / "metadata.csv")
metadata = metadata[metadata["emotion"].isin(COMMON_EMOTIONS)].copy()
if "readable" in metadata.columns:
    metadata = metadata[metadata["readable"] == True].copy()

metadata["label_id"] = metadata["emotion"].map(LABEL_TO_ID).astype(int)
metadata["speaker_id"] = metadata["speaker_id"].astype(str)
metadata["domain_id"] = metadata["dataset"].astype("category").cat.codes.astype(int)
DOMAIN_NAMES = list(metadata["dataset"].astype("category").cat.categories)
NUM_DOMAINS = len(DOMAIN_NAMES)

if QUICK_RUN:
    metadata = (
        metadata.sort_values(["dataset", "emotion", "sample_id"])
        .groupby(["dataset", "emotion"], group_keys=False)
        .head(QUICK_RUN_PER_GROUP)
        .reset_index(drop=True)
    )
    metadata["domain_id"] = metadata["dataset"].astype("category").cat.codes.astype(int)
    DOMAIN_NAMES = list(metadata["dataset"].astype("category").cat.categories)
    NUM_DOMAINS = len(DOMAIN_NAMES)
else:
    metadata = metadata.reset_index(drop=True)

metadata["split_strict_original"] = metadata["split"].astype(str).str.lower() if "split" in metadata.columns else "train"
metadata.loc[~metadata["split_strict_original"].isin(["train", "validation", "test"]), "split_strict_original"] = "train"

metadata["strict_include"] = metadata["dataset"] != "TESS"
metadata["split_strict_project"] = metadata["split_strict_original"]
savee_plan = {"savee_DC": "train", "savee_JE": "train", "savee_JK": "validation", "savee_KL": "test"}
savee_mask = metadata["dataset"].eq("SAVEE")
metadata.loc[savee_mask, "split_strict_project"] = metadata.loc[savee_mask, "speaker_id"].map(savee_plan).fillna("train")
metadata.loc[metadata["dataset"].eq("TESS"), "split_strict_project"] = "excluded"

train_idx, temp_idx = train_test_split(
    metadata.index,
    test_size=0.30,
    random_state=SEED,
    stratify=metadata["label_id"],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=metadata.loc[temp_idx, "label_id"],
)
metadata["split_random"] = "train"
metadata.loc[val_idx, "split_random"] = "validation"
metadata.loc[test_idx, "split_random"] = "test"

display(metadata.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))
display(metadata.groupby(["dataset", "split_strict_project"]).size().unstack(fill_value=0))

metadata.groupby(["dataset", "emotion"]).size().reset_index(name="count").to_csv(
    REPORT_DIR / "dataset_emotion_distribution.csv", index=False
)
metadata.groupby(["dataset", "split_strict_project"]).size().reset_index(name="count").to_csv(
    REPORT_DIR / "project_strict_split_by_dataset.csv", index=False
)


## 6. Feature Extraction

### Branch A temporal sequence

\[
X_{temporal} \in \mathbb{R}^{N \times T \times 132}
\]

Gồm:

```text
40 MFCC + 40 delta + 40 delta-delta
+ RMS + ZCR + centroid + bandwidth + rolloff + 7 spectral contrast
= 132 features/frame
```

### Branch B spectral tensor

\[
X_{spectral} \in \mathbb{R}^{N \times 3 \times M \times T}
\]

3 channel:

```text
log-Mel
delta log-Mel
delta-delta log-Mel
```

### Branch D statistical vector

Tính nhiều thống kê theo thời gian:

```text
mean, std, min, max, median, p25, p75, IQR, range, skewness, kurtosis
```


In [ ]:
def center_crop_or_pad(y, target_length=TARGET_LENGTH):
    if len(y) > target_length:
        start = (len(y) - target_length) // 2
        y = y[start:start + target_length]
    elif len(y) < target_length:
        pad = target_length - len(y)
        y = np.pad(y, (pad // 2, pad - pad // 2), mode="constant")
    return y.astype(np.float32)

def normalize_audio(y):
    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y)
    peak = np.max(np.abs(y)) + 1e-8
    if peak > 1.0:
        y = y / peak
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def read_audio(row):
    cached = AUDIO_16K_DIR / f"{row.sample_id}.wav"
    if cached.exists():
        y, sr = sf.read(cached)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
    else:
        y, sr = librosa.load(row.filepath, sr=TARGET_SR, mono=True)
    return normalize_audio(center_crop_or_pad(y, TARGET_LENGTH))

def random_waveform_augment(y):
    if not USE_WAVEFORM_AUGMENTATION:
        return y.astype(np.float32)
    y_aug = y.astype(np.float32).copy()
    if random.random() < 0.35:
        try:
            y_aug = librosa.effects.pitch_shift(y_aug, sr=TARGET_SR, n_steps=random.uniform(-2.0, 2.0))
        except Exception:
            pass
    if random.random() < 0.30:
        try:
            y_aug = librosa.effects.time_stretch(y_aug, rate=random.uniform(0.88, 1.12))
            y_aug = center_crop_or_pad(y_aug, TARGET_LENGTH)
        except Exception:
            pass
    if random.random() < 0.45:
        rms = np.sqrt(np.mean(y_aug ** 2) + 1e-8)
        snr_db = random.uniform(14.0, 28.0)
        noise_rms = rms / (10 ** (snr_db / 20.0))
        y_aug = y_aug + np.random.normal(0.0, noise_rms, size=len(y_aug)).astype(np.float32)
    if random.random() < 0.35:
        y_aug *= random.uniform(0.72, 1.28)
    if random.random() < 0.35:
        shift = random.randint(-int(0.18 * TARGET_SR), int(0.18 * TARGET_SR))
        y_aug = np.roll(y_aug, shift)
        if shift > 0:
            y_aug[:shift] = 0
        elif shift < 0:
            y_aug[shift:] = 0
    if random.random() < 0.20:
        width = random.randint(int(0.035 * TARGET_SR), int(0.14 * TARGET_SR))
        start = random.randint(0, max(0, len(y_aug) - width))
        y_aug[start:start + width] *= random.uniform(0.0, 0.15)
    return normalize_audio(center_crop_or_pad(y_aug, TARGET_LENGTH))

def pad_time(x, target, axis=-1):
    if x.shape[axis] > target:
        sl = [slice(None)] * x.ndim
        sl[axis] = slice(0, target)
        return x[tuple(sl)]
    if x.shape[axis] < target:
        pad_width = [(0, 0)] * x.ndim
        pad_width[axis] = (0, target - x.shape[axis])
        return np.pad(x, pad_width, mode="constant")
    return x

def safe_skew_kurtosis(x, axis=1):
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True) + 1e-6
    z = (x - mean) / std
    skew = np.mean(z ** 3, axis=axis)
    kurt = np.mean(z ** 4, axis=axis) - 3.0
    return skew, kurt

def stats_full(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        x = x[None, :]
    mean = x.mean(axis=1)
    std = x.std(axis=1)
    mn = x.min(axis=1)
    mx = x.max(axis=1)
    med = np.median(x, axis=1)
    p25 = np.percentile(x, 25, axis=1)
    p75 = np.percentile(x, 75, axis=1)
    iqr = p75 - p25
    rg = mx - mn
    skew, kurt = safe_skew_kurtosis(x, axis=1)
    return np.concatenate([mean, std, mn, mx, med, p25, p75, iqr, rg, skew, kurt]).astype(np.float32)

def extract_pitch_stats(y):
    if not ENABLE_RICH_PITCH:
        return np.zeros(11, dtype=np.float32)
    try:
        f0 = librosa.yin(y, fmin=50, fmax=500, sr=TARGET_SR, frame_length=max(1024, N_FFT), hop_length=HOP_LENGTH)
        f0 = f0[np.isfinite(f0)]
        f0 = f0[(f0 >= 50) & (f0 <= 500)]
        if len(f0) < 3:
            return np.zeros(11, dtype=np.float32)
        return stats_full(f0).astype(np.float32)
    except Exception:
        return np.zeros(11, dtype=np.float32)

def frame_entropy(power):
    p = power / (np.sum(power, axis=0, keepdims=True) + 1e-8)
    return (-np.sum(p * np.log(p + 1e-8), axis=0, keepdims=True)).astype(np.float32)

def extract_representations(row=None, y_override=None):
    y_audio = read_audio(row) if y_override is None else normalize_audio(center_crop_or_pad(y_override, TARGET_LENGTH))

    mfcc = librosa.feature.mfcc(y=y_audio, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    rms = librosa.feature.rms(y=y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    zcr = librosa.feature.zero_crossing_rate(y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    centroid = librosa.feature.spectral_centroid(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    bandwidth = librosa.feature.spectral_bandwidth(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rolloff = librosa.feature.spectral_rolloff(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_bands=6)

    mel = librosa.feature.melspectrogram(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, n_mels=N_MELS, power=2.0)
    logmel = librosa.power_to_db(mel, ref=np.max)
    d_logmel = librosa.feature.delta(logmel)
    d2_logmel = librosa.feature.delta(logmel, order=2)

    chroma_stft = librosa.feature.chroma_stft(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    chroma_cqt = librosa.feature.chroma_cqt(y=y_audio, sr=TARGET_SR, hop_length=HOP_LENGTH)
    chroma_cens = librosa.feature.chroma_cens(y=y_audio, sr=TARGET_SR, hop_length=HOP_LENGTH)
    power = np.abs(librosa.stft(y_audio, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)) ** 2
    energy = power.sum(axis=0, keepdims=True)
    entropy = frame_entropy(power)

    temporal = np.concatenate([
        pad_time(mfcc, MAX_FRAMES, axis=1),
        pad_time(delta, MAX_FRAMES, axis=1),
        pad_time(delta2, MAX_FRAMES, axis=1),
        pad_time(rms, MAX_FRAMES, axis=1),
        pad_time(zcr, MAX_FRAMES, axis=1),
        pad_time(centroid, MAX_FRAMES, axis=1),
        pad_time(bandwidth, MAX_FRAMES, axis=1),
        pad_time(rolloff, MAX_FRAMES, axis=1),
        pad_time(contrast, MAX_FRAMES, axis=1),
    ], axis=0).T.astype(np.float32)

    spectral = np.stack([
        pad_time(logmel, MAX_FRAMES, axis=1),
        pad_time(d_logmel, MAX_FRAMES, axis=1),
        pad_time(d2_logmel, MAX_FRAMES, axis=1),
    ], axis=0).astype(np.float32)

    stat_vec = np.concatenate([
        stats_full(mfcc), stats_full(delta), stats_full(delta2),
        stats_full(chroma_stft), stats_full(chroma_cqt), stats_full(chroma_cens),
        stats_full(centroid), stats_full(bandwidth), stats_full(rolloff), stats_full(contrast),
        stats_full(rms), stats_full(zcr), stats_full(energy), stats_full(entropy), stats_full(logmel),
        extract_pitch_stats(y_audio),
    ]).astype(np.float32)

    return temporal, spectral, stat_vec, y_audio


In [ ]:
cache_name = (
    f"advanced_features_mfcc{N_MFCC}_mel{N_MELS}_"
    f"{int(TARGET_DURATION)}s_{TARGET_SR}hz_pitch{int(ENABLE_RICH_PITCH)}_{len(metadata)}files.npz"
)
cache_path = CACHE_DIR / cache_name

if CACHE_FEATURES and cache_path.exists():
    data = np.load(cache_path)
    X_temporal = data["X_temporal"].astype(np.float32)
    X_spectral = data["X_spectral"].astype(np.float32)
    X_stats = data["X_stats"].astype(np.float32)
    y = data["y"].astype(np.int64)
    domain_y = data["domain_y"].astype(np.int64)
    print("Loaded feature cache:", cache_path)
else:
    X_temporal, X_spectral, X_stats = [], [], []
    start = time.time()
    for i, row in enumerate(metadata.itertuples(index=False)):
        a, b, c, _ = extract_representations(row)
        X_temporal.append(a)
        X_spectral.append(b)
        X_stats.append(c)
        if (i + 1) % 250 == 0:
            print(f"Extracted {i+1}/{len(metadata)} files in {(time.time()-start)/60:.1f} min")
    X_temporal = np.stack(X_temporal).astype(np.float32)
    X_spectral = np.stack(X_spectral).astype(np.float32)
    X_stats = np.stack(X_stats).astype(np.float32)
    y = metadata["label_id"].to_numpy(np.int64)
    domain_y = metadata["domain_id"].to_numpy(np.int64)
    if CACHE_FEATURES:
        np.savez_compressed(cache_path, X_temporal=X_temporal, X_spectral=X_spectral, X_stats=X_stats, y=y, domain_y=domain_y)
        print("Saved feature cache:", cache_path)

TEMPORAL_DIM = X_temporal.shape[-1]
STATS_DIM = X_stats.shape[-1]
print("X_temporal:", X_temporal.shape)
print("X_spectral:", X_spectral.shape)
print("X_stats:", X_stats.shape)
print("domain names:", DOMAIN_NAMES)


## 7. Train-Only Waveform Augmented Cache

Cache này có thể được tạo cho toàn metadata để tái sử dụng, nhưng khi train chỉ lấy index thuộc `train_idx`. Validation/test không dùng augmented feature.


In [ ]:
if USE_WAVEFORM_AUGMENTATION and WAVEFORM_AUGMENT_COPIES > 0:
    aug_cache_name = (
        f"advanced_wave_aug_copies{WAVEFORM_AUGMENT_COPIES}_"
        f"mfcc{N_MFCC}_mel{N_MELS}_{int(TARGET_DURATION)}s_{len(metadata)}files.npz"
    )
    aug_cache_path = CACHE_DIR / aug_cache_name
    if CACHE_FEATURES and aug_cache_path.exists():
        aug_data = np.load(aug_cache_path)
        X_temporal_wave_aug = aug_data["X_temporal_wave_aug"].astype(np.float32)
        X_spectral_wave_aug = aug_data["X_spectral_wave_aug"].astype(np.float32)
        X_stats_wave_aug = aug_data["X_stats_wave_aug"].astype(np.float32)
        print("Loaded waveform augmentation cache:", aug_cache_path)
    else:
        aug_temporal, aug_spectral, aug_stats = [], [], []
        start = time.time()
        for copy_i in range(WAVEFORM_AUGMENT_COPIES):
            copy_temporal, copy_spectral, copy_stats = [], [], []
            for i, row in enumerate(metadata.itertuples(index=False)):
                y_base = read_audio(row)
                y_aug = random_waveform_augment(y_base)
                a, b, c, _ = extract_representations(row, y_override=y_aug)
                copy_temporal.append(a)
                copy_spectral.append(b)
                copy_stats.append(c)
                if (i + 1) % 250 == 0:
                    print(f"Wave aug copy {copy_i+1}/{WAVEFORM_AUGMENT_COPIES}: {i+1}/{len(metadata)} in {(time.time()-start)/60:.1f} min")
            aug_temporal.append(np.stack(copy_temporal).astype(np.float32))
            aug_spectral.append(np.stack(copy_spectral).astype(np.float32))
            aug_stats.append(np.stack(copy_stats).astype(np.float32))
        X_temporal_wave_aug = np.stack(aug_temporal).astype(np.float32)
        X_spectral_wave_aug = np.stack(aug_spectral).astype(np.float32)
        X_stats_wave_aug = np.stack(aug_stats).astype(np.float32)
        if CACHE_FEATURES:
            np.savez_compressed(aug_cache_path, X_temporal_wave_aug=X_temporal_wave_aug, X_spectral_wave_aug=X_spectral_wave_aug, X_stats_wave_aug=X_stats_wave_aug)
            print("Saved waveform augmentation cache:", aug_cache_path)
else:
    X_temporal_wave_aug = None
    X_spectral_wave_aug = None
    X_stats_wave_aug = None
    print("Waveform augmentation cache disabled.")

if X_temporal_wave_aug is not None:
    print("X_temporal_wave_aug:", X_temporal_wave_aug.shape)
    print("X_spectral_wave_aug:", X_spectral_wave_aug.shape)
    print("X_stats_wave_aug:", X_stats_wave_aug.shape)


## 8. WavLM Frozen Embeddings

Branch C dùng raw waveform đưa qua WavLM.

Trong notebook:

- WavLM frozen: không fine-tune backbone.
- Mean pooling theo frame để tạo utterance embedding.
- Adapter MLP sẽ học emotion trên embedding đó.


In [ ]:
def extract_pretrained_speech_embeddings():
    if not RUN_PRETRAINED_SPEECH:
        print("Pretrained speech branch disabled.")
        return None
    embed_cache = CACHE_DIR / ("speech_embed_" + PRETRAINED_SPEECH_MODEL.replace("/", "_") + f"_{int(TARGET_DURATION)}s_{len(metadata)}files.npz")
    if CACHE_FEATURES and embed_cache.exists():
        arr = np.load(embed_cache)["X_speech_embed"].astype(np.float32)
        print("Loaded speech embedding cache:", embed_cache, arr.shape)
        return arr
    try:
        from transformers import AutoFeatureExtractor, AutoModel
    except Exception as e:
        print("Cannot import transformers; skip pretrained speech branch:", e)
        return None
    try:
        feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_SPEECH_MODEL)
        encoder = AutoModel.from_pretrained(PRETRAINED_SPEECH_MODEL).to(DEVICE)
        encoder = maybe_data_parallel(encoder, name="pretrained speech encoder")
        encoder.eval()
    except Exception as e:
        print("Cannot load pretrained speech model; skip branch:", e)
        return None
    embeddings = []
    start = time.time()
    with torch.no_grad():
        for start_i in range(0, len(metadata), SPEECH_EMBED_BATCH_SIZE):
            rows = metadata.iloc[start_i:start_i + SPEECH_EMBED_BATCH_SIZE]
            waves = [read_audio(row) for row in rows.itertuples(index=False)]
            inputs = feature_extractor(waves, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            out = encoder(**inputs)
            hidden = out.last_hidden_state
            if "attention_mask" in inputs:
                mask = inputs["attention_mask"]
                frame_mask = F.interpolate(mask[:, None, :].float(), size=hidden.shape[1], mode="nearest").squeeze(1)
                pooled = (hidden * frame_mask[..., None]).sum(dim=1) / (frame_mask.sum(dim=1, keepdim=True) + 1e-8)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy().astype(np.float32))
            done = min(start_i + SPEECH_EMBED_BATCH_SIZE, len(metadata))
            if done % 240 == 0 or done == len(metadata):
                print(f"Speech embeddings {done}/{len(metadata)} in {(time.time()-start)/60:.1f} min")
    X_speech_embed = np.concatenate(embeddings, axis=0).astype(np.float32)
    if CACHE_FEATURES:
        np.savez_compressed(embed_cache, X_speech_embed=X_speech_embed)
        print("Saved speech embedding cache:", embed_cache)
    return X_speech_embed

X_speech_embed = extract_pretrained_speech_embeddings()
if X_speech_embed is not None:
    print("X_speech_embed:", X_speech_embed.shape)


## 9. Dataset, Scaling and Train-Only Augmentation

Scaler luôn fit trên train split:

\[
\mu, \sigma = \mathrm{mean/std}(X_{train})
\]

Validation/test chỉ transform bằng train scaler.

SpecAugment áp dụng cho spectrogram train:

- Time mask: che một khoảng thời gian.
- Frequency mask: che một vùng mel bins.


In [ ]:
def compute_scalers(train_idx):
    scalers = {
        "temporal_mean": X_temporal[train_idx].mean(axis=(0, 1), keepdims=True).astype(np.float32),
        "temporal_std": (X_temporal[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6).astype(np.float32),
        # Keep spectral scaler as [C, 1, 1]. If keepdims=True is used here,
        # normalizing one sample [C, mel, frames] broadcasts to [1, C, mel, frames].
        "spectral_mean": X_spectral[train_idx].mean(axis=(0, 2, 3)).reshape(X_spectral.shape[1], 1, 1).astype(np.float32),
        "spectral_std": (X_spectral[train_idx].std(axis=(0, 2, 3)) + 1e-6).reshape(X_spectral.shape[1], 1, 1).astype(np.float32),
        "stats_scaler": StandardScaler().fit(X_stats[train_idx]),
        "speech_scaler": StandardScaler().fit(X_speech_embed[train_idx]) if X_speech_embed is not None else None,
    }
    return scalers

def augment_temporal(x):
    if not USE_AUGMENTATION:
        return x
    x = x.copy()
    if random.random() < 0.30:
        x = np.roll(x, random.randint(-18, 18), axis=0)
    if random.random() < 0.35:
        x += np.random.normal(0, 0.02, x.shape).astype(np.float32)
    if random.random() < 0.35:
        width = random.randint(8, 36)
        start = random.randint(0, max(0, x.shape[0] - width))
        x[start:start + width, :] = 0
    if random.random() < 0.25:
        width = random.randint(4, 18)
        start = random.randint(0, max(0, x.shape[1] - width))
        x[:, start:start + width] = 0
    return x.astype(np.float32)

def augment_spectral(x):
    if not USE_AUGMENTATION or random.random() > SPECTRAL_AUG_PROB:
        return x
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 4 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 3:
        raise ValueError(f"augment_spectral expects [C, mel, frames], got {x.shape}")
    x = x.copy()
    _, mels, frames = x.shape
    if random.random() < 0.70:
        width = random.randint(8, min(40, frames))
        start = random.randint(0, max(0, frames - width))
        x[:, :, start:start + width] = 0
    if random.random() < 0.70:
        width = random.randint(4, min(18, mels))
        start = random.randint(0, max(0, mels - width))
        x[:, start:start + width, :] = 0
    if random.random() < 0.30:
        x += np.random.normal(0, 0.02, x.shape).astype(np.float32)
    return x.astype(np.float32)

class MultiBranchDataset(Dataset):
    def __init__(self, indices, scalers, train=False):
        self.indices = np.array(indices)
        self.scalers = scalers
        self.train = train
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        use_wave_aug = self.train and X_temporal_wave_aug is not None and random.random() < TEMPORAL_WAVE_AUG_PROB
        if use_wave_aug:
            copy_i = random.randrange(X_temporal_wave_aug.shape[0])
            temporal_raw = X_temporal_wave_aug[copy_i, i]
            spectral_raw = X_spectral_wave_aug[copy_i, i]
            stats_raw = X_stats_wave_aug[copy_i, i]
        else:
            temporal_raw = X_temporal[i]
            spectral_raw = X_spectral[i]
            stats_raw = X_stats[i]

        temporal = ((temporal_raw - self.scalers["temporal_mean"].squeeze(0)) / self.scalers["temporal_std"].squeeze(0)).astype(np.float32)
        spectral = ((spectral_raw - self.scalers["spectral_mean"]) / self.scalers["spectral_std"]).astype(np.float32)
        if spectral.ndim == 4 and spectral.shape[0] == 1:
            spectral = spectral[0]
        if spectral.ndim != 3:
            raise ValueError(f"spectral sample must be [C, mel, frames], got {spectral.shape}")
        stats = self.scalers["stats_scaler"].transform(stats_raw[None, :]).astype(np.float32)[0]
        if self.train:
            temporal = augment_temporal(temporal)
            spectral = augment_spectral(spectral)
        if X_speech_embed is not None:
            speech = self.scalers["speech_scaler"].transform(X_speech_embed[i:i+1]).astype(np.float32)[0]
        else:
            speech = np.zeros(1, dtype=np.float32)
        return {
            "temporal": torch.from_numpy(temporal),
            "spectral": torch.from_numpy(spectral),
            "stats": torch.from_numpy(stats),
            "speech": torch.from_numpy(speech),
            "label": torch.tensor(y[i], dtype=torch.long),
            "domain": torch.tensor(domain_y[i], dtype=torch.long),
        }

def make_sampler(indices):
    if not USE_BALANCED_SAMPLER:
        return None
    labels = y[indices]
    datasets = metadata.iloc[indices]["dataset"].to_numpy()
    key_counts = defaultdict(int)
    for ds, lab in zip(datasets, labels):
        key_counts[(ds, int(lab))] += 1
    weights = np.array([1.0 / key_counts[(ds, int(lab))] for ds, lab in zip(datasets, labels)], dtype=np.float32)
    weights = weights / weights.mean()
    return WeightedRandomSampler(torch.from_numpy(weights), num_samples=len(indices), replacement=True)

def make_loader(dataset, indices, train=False):
    sampler = make_sampler(indices) if train else None
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(train and sampler is None), sampler=sampler, num_workers=0, pin_memory=torch.cuda.is_available())


## 10. Model Blocks

### Branch A

`TemporalCNNBiLSTMBranch`:

```text
[B, T, F] -> LayerNorm -> 1D-CNN -> BiLSTM -> attention pooling -> z_temporal
```

### Branch B

`SpectralCNNSEBranch`:

```text
[B, 3, n_mels, T] -> 2D residual SE blocks -> global pooling -> z_spectral
```

### Branch C

`SpeechAdapterBranch`:

```text
WavLM embedding -> MLP adapter -> z_wavlm
```

### Branch D

`StatsMLPBranch`:

```text
statistical vector -> MLP -> z_stats
```

### Fusion

```text
concat(z_temporal, z_spectral, z_wavlm, z_stats)
-> emotion head
-> optional GRL -> domain head
```


In [ ]:
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradientReversalFn.apply(x, lambd)

class AttentionPoolSeq(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1), w.squeeze(-1)

class TemporalCNNBiLSTMBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=192, hidden=128, dropout=DROPOUT):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Conv1d(128, 160, kernel_size=5, padding=4, dilation=2),
            nn.BatchNorm1d(160), nn.GELU(), nn.Dropout(dropout * 0.5),
        )
        self.lstm = nn.LSTM(input_size=160, hidden_size=hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.pool = AttentionPoolSeq(hidden * 2)
        self.proj = nn.Sequential(nn.LayerNorm(hidden * 2), nn.Linear(hidden * 2, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        x = self.norm(x)
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(x)
        pooled, attn = self.pool(out)
        return self.proj(pooled)

class SEBlock2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(4, channels // reduction)
        self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, mid), nn.GELU(), nn.Linear(mid, channels), nn.Sigmoid())
    def forward(self, x):
        w = self.net(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class ResidualSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock2D(out_ch)
        self.skip = nn.Identity() if in_ch == out_ch and stride == 1 else nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
        self.act = nn.GELU()
        self.drop = nn.Dropout2d(dropout * 0.4)
    def forward(self, x):
        out = self.conv(x)
        out = self.se(out)
        out = self.act(out + self.skip(x))
        return self.drop(out)

class SpectralCNNSEBranch(nn.Module):
    def __init__(self, in_ch=3, emb_dim=192, dropout=DROPOUT):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 48, 5, stride=2, padding=2, bias=False), nn.BatchNorm2d(48), nn.GELU())
        self.blocks = nn.Sequential(
            ResidualSEBlock(48, 64, stride=1, dropout=dropout),
            ResidualSEBlock(64, 96, stride=2, dropout=dropout),
            ResidualSEBlock(96, 128, stride=2, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Flatten(), nn.LayerNorm(128), nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x)
        return self.proj(x)

class SpeechAdapterBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=160, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 384), nn.LayerNorm(384), nn.GELU(), nn.Dropout(dropout), nn.Linear(384, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class StatsMLPBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=128, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class AdvancedDomainRobustSER(nn.Module):
    def __init__(self, temporal_dim, spectral_channels, stats_dim, speech_dim, num_classes, num_domains):
        super().__init__()
        self.temporal = TemporalCNNBiLSTMBranch(temporal_dim)
        self.spectral = SpectralCNNSEBranch(spectral_channels)
        self.use_speech = speech_dim > 1
        self.speech = SpeechAdapterBranch(speech_dim) if self.use_speech else None
        self.stats = StatsMLPBranch(stats_dim)
        fusion_dim = 192 + 192 + 128 + (160 if self.use_speech else 0)
        self.fusion = nn.Sequential(nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 256), nn.GELU(), nn.Dropout(DROPOUT))
        self.emotion_head = nn.Linear(256, num_classes)
        self.domain_head = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_domains))
    def forward(self, temporal, spectral, stats, speech, grl_lambda=0.0):
        parts = [self.temporal(temporal), self.spectral(spectral), self.stats(stats)]
        if self.use_speech:
            parts.append(self.speech(speech))
        fused = self.fusion(torch.cat(parts, dim=1))
        emo_logits = self.emotion_head(fused)
        dom_logits = self.domain_head(grad_reverse(fused, grl_lambda)) if grl_lambda > 0 else self.domain_head(fused.detach())
        return emo_logits, dom_logits, fused


## 11. Training Logic

Adversarial schedule:

\[
\lambda(e) =
\begin{cases}
0, & e \leq E_{warmup} \\
\lambda_{max} \cdot \frac{e - E_{warmup}}{E_{total} - E_{warmup}}, & e > E_{warmup}
\end{cases}
\]

Mục đích: để model học emotion trước, sau đó mới ép giảm domain signature nhẹ.


In [ ]:
def class_weights_for(indices):
    counts = np.bincount(y[indices], minlength=len(COMMON_EMOTIONS)).astype(np.float32)
    weights = counts.sum() / (len(COMMON_EMOTIONS) * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def domain_weights_for(indices):
    counts = np.bincount(domain_y[indices], minlength=NUM_DOMAINS).astype(np.float32)
    weights = counts.sum() / (NUM_DOMAINS * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def adv_lambda_for_epoch(epoch):
    if not USE_DOMAIN_ADVERSARIAL or epoch <= ADV_WARMUP_EPOCHS:
        return 0.0
    denom = max(1, MAX_EPOCHS - ADV_WARMUP_EPOCHS)
    progress = min(1.0, (epoch - ADV_WARMUP_EPOCHS) / denom)
    return ADV_LAMBDA_MAX * progress

def move_batch(batch):
    return {k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}

def mixup_tensors(batch, alpha=MIXUP_ALPHA):
    if alpha <= 0 or random.random() > MIXUP_PROB:
        return batch, batch["label"], None, None
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(batch["label"].size(0), device=batch["label"].device)
    mixed = dict(batch)
    for key in ["temporal", "spectral", "stats", "speech"]:
        mixed[key] = lam * batch[key] + (1 - lam) * batch[key][index]
    return mixed, batch["label"], batch["label"][index], lam

def mixup_ce(logits, y_a, y_b, lam, criterion):
    if y_b is None:
        return criterion(logits, y_a)
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

def run_epoch(model, loader, emotion_criterion, domain_criterion, optimizer=None, epoch=1):
    is_train = optimizer is not None
    model.train(is_train)
    grl_lambda = adv_lambda_for_epoch(epoch) if is_train else 0.0
    total_loss, total_emo_loss, total_dom_loss = 0.0, 0.0, 0.0
    all_y, all_pred, all_prob = [], [], []
    all_dom, all_dom_pred = [], []
    start = time.time()

    for batch in loader:
        batch = move_batch(batch)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            batch_in, y_a, y_b, lam = mixup_tensors(batch)
        else:
            batch_in, y_a, y_b, lam = batch, batch["label"], None, None

        with torch.set_grad_enabled(is_train):
            emo_logits, dom_logits, _ = model(batch_in["temporal"], batch_in["spectral"], batch_in["stats"], batch_in["speech"], grl_lambda=grl_lambda)
            emo_loss = mixup_ce(emo_logits, y_a, y_b, lam, emotion_criterion)
            dom_loss = domain_criterion(dom_logits, batch["domain"]) if grl_lambda > 0 else torch.tensor(0.0, device=DEVICE)
            loss = emo_loss + DOMAIN_LOSS_WEIGHT * dom_loss
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        prob = torch.softmax(emo_logits.detach(), dim=1)
        dom_pred = dom_logits.detach().argmax(dim=1)
        n = len(batch["label"])
        total_loss += loss.item() * n
        total_emo_loss += emo_loss.item() * n
        total_dom_loss += dom_loss.item() * n
        all_y.extend(batch["label"].detach().cpu().numpy())
        all_pred.extend(prob.argmax(dim=1).cpu().numpy())
        all_prob.extend(prob.cpu().numpy())
        all_dom.extend(batch["domain"].detach().cpu().numpy())
        all_dom_pred.extend(dom_pred.cpu().numpy())

    all_y = np.array(all_y)
    all_pred = np.array(all_pred)
    all_prob = np.array(all_prob)
    all_dom = np.array(all_dom)
    all_dom_pred = np.array(all_dom_pred)
    elapsed = time.time() - start
    return {
        "loss": total_loss / max(1, len(all_y)),
        "emotion_loss": total_emo_loss / max(1, len(all_y)),
        "domain_loss": total_dom_loss / max(1, len(all_y)),
        "accuracy": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_y, all_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(all_y, all_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(all_y, all_pred, average="macro", zero_division=0),
        "domain_accuracy": accuracy_score(all_dom, all_dom_pred) if len(all_dom) else 0.0,
        "y_true": all_y,
        "y_pred": all_pred,
        "prob": all_prob,
        "elapsed_sec": elapsed,
        "grl_lambda": grl_lambda,
    }

def train_advanced_model(run_name, split_map):
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]
    scalers = compute_scalers(train_idx)
    train_ds = MultiBranchDataset(train_idx, scalers, train=True)
    val_ds = MultiBranchDataset(val_idx, scalers, train=False)
    test_ds = MultiBranchDataset(test_idx, scalers, train=False)
    train_loader = make_loader(train_ds, train_idx, train=True)
    val_loader = make_loader(val_ds, val_idx, train=False)
    test_loader = make_loader(test_ds, test_idx, train=False)

    speech_dim = X_speech_embed.shape[1] if X_speech_embed is not None else 1
    model = AdvancedDomainRobustSER(TEMPORAL_DIM, X_spectral.shape[1], STATS_DIM, speech_dim, len(COMMON_EMOTIONS), NUM_DOMAINS)
    model = maybe_data_parallel(model.to(DEVICE), name=f"{run_name}/advanced_domain_robust_ser")

    emotion_weights = class_weights_for(train_idx) if USE_CLASS_WEIGHTS else None
    domain_weights = domain_weights_for(train_idx)
    emotion_criterion = nn.CrossEntropyLoss(weight=emotion_weights, label_smoothing=LABEL_SMOOTHING)
    domain_criterion = nn.CrossEntropyLoss(weight=domain_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    history = []
    best_state, best_epoch, best_val = None, 0, -1
    patience_left = PATIENCE
    train_start = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_m = run_epoch(model, train_loader, emotion_criterion, domain_criterion, optimizer=optimizer, epoch=epoch)
        val_m = run_epoch(model, val_loader, emotion_criterion, domain_criterion, epoch=epoch)
        scheduler.step(val_m["macro_f1"])
        row = {
            "run_name": run_name,
            "model": "advanced_domain_robust_ser",
            "epoch": epoch,
            "train_macro_f1": train_m["macro_f1"],
            "val_macro_f1": val_m["macro_f1"],
            "train_loss": train_m["loss"],
            "val_loss": val_m["loss"],
            "train_domain_acc": train_m["domain_accuracy"],
            "val_domain_acc": val_m["domain_accuracy"],
            "grl_lambda": train_m["grl_lambda"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(
            f"{run_name} | epoch {epoch:02d} | train f1 {train_m['macro_f1']:.4f} "
            f"| val f1 {val_m['macro_f1']:.4f} | domain acc {train_m['domain_accuracy']:.4f} "
            f"| grl {train_m['grl_lambda']:.4f}"
        )
        if val_m["macro_f1"] > best_val:
            best_val = val_m["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping.")
                break

    train_time = time.time() - train_start
    model.load_state_dict(best_state)
    val_final = run_epoch(model, val_loader, emotion_criterion, domain_criterion, epoch=best_epoch)
    test_final = run_epoch(model, test_loader, emotion_criterion, domain_criterion, epoch=best_epoch)
    return model, scalers, {"history": history, "val": val_final, "test": test_final, "best_epoch": best_epoch, "best_val_macro_f1": best_val, "train_time_sec": train_time}


## 12. RBF-SVM Statistical Branch

SVM không học chuỗi thời gian, nhưng phù hợp với statistical vector cố định.

Pipeline:

```text
VarianceThreshold -> StandardScaler -> PCA -> RBF-SVM(probability=True)
```

RBF kernel:

\[
K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)
\]


In [ ]:
def fit_stats_svm(split_map):
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]
    start = time.time()
    n_components = min(STATS_PCA_COMPONENTS, max(2, len(train_idx) - 1), X_stats.shape[1])
    model = make_pipeline(
        VarianceThreshold(),
        StandardScaler(),
        PCA(n_components=n_components, random_state=SEED),
        SVC(kernel="rbf", C=8.0, gamma="scale", probability=True, class_weight="balanced", random_state=SEED),
    )
    X_train_stats = X_stats[train_idx]
    y_train_stats = y[train_idx]
    if X_stats_wave_aug is not None:
        aug_x = X_stats_wave_aug[:, train_idx, :].reshape(-1, X_stats.shape[1])
        aug_y = np.tile(y[train_idx], X_stats_wave_aug.shape[0])
        X_train_stats = np.vstack([X_train_stats, aug_x]).astype(np.float32)
        y_train_stats = np.concatenate([y_train_stats, aug_y]).astype(np.int64)
    model.fit(X_train_stats, y_train_stats)
    train_time = time.time() - start
    def predict(indices):
        t0 = time.time()
        prob = model.predict_proba(X_stats[indices])
        elapsed = time.time() - t0
        pred = prob.argmax(axis=1)
        yy = y[indices]
        return {
            "accuracy": accuracy_score(yy, pred),
            "macro_f1": f1_score(yy, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(yy, pred, average="weighted", zero_division=0),
            "uar": balanced_accuracy_score(yy, pred),
            "macro_precision": precision_score(yy, pred, average="macro", zero_division=0),
            "macro_recall": recall_score(yy, pred, average="macro", zero_division=0),
            "y_true": yy,
            "y_pred": pred,
            "prob": prob,
            "elapsed_sec": elapsed,
        }
    return model, {"val": predict(val_idx), "test": predict(test_idx), "train_time_sec": train_time}


## 13. Protocol Builder


In [ ]:
def split_map_from_column(column, include_mask=None):
    df = metadata if include_mask is None else metadata[include_mask].copy()
    return {
        "train": df.index[df[column].eq("train")].to_numpy(),
        "validation": df.index[df[column].eq("validation")].to_numpy(),
        "test": df.index[df[column].eq("test")].to_numpy(),
    }

def random_single_dataset_split(dataset_name):
    idx = metadata.index[metadata["dataset"].eq(dataset_name)].to_numpy()
    labels = metadata.loc[idx, "label_id"]
    tr, temp = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=labels)
    va, te = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=metadata.loc[temp, "label_id"])
    return {"train": tr, "validation": va, "test": te}

protocols = []
if RUN_COMBINED_RANDOM:
    protocols.append(("combined_random", split_map_from_column("split_random")))
if RUN_COMBINED_STRICT:
    strict_mask = metadata["strict_include"] & metadata["split_strict_project"].isin(["train", "validation", "test"])
    protocols.append(("combined_strict_no_tess", split_map_from_column("split_strict_project", strict_mask)))
if RUN_SINGLE_DATASET:
    for ds in sorted(metadata["dataset"].unique()):
        protocols.append((f"single_{ds}", random_single_dataset_split(ds)))

for name, smap in protocols:
    print(name, {k: len(v) for k, v in smap.items()})


## 14. Run Experiments

Mỗi protocol:

1. Train advanced deep fusion model.
2. Train RBF-SVM statistical branch.
3. Train stacking logistic regression trên validation probabilities.
4. Evaluate trên test set.


In [ ]:
def metrics_row(run_name, model_name, split_name, result, n_samples):
    return {
        "run_name": run_name,
        "model": model_name,
        "split": split_name,
        "n_samples": n_samples,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "inference_time_sec": result.get("elapsed_sec", 0.0),
        "inference_ms_per_sample": 1000 * result.get("elapsed_sec", 0.0) / max(1, n_samples),
    }

all_metrics, all_history, all_predictions = [], [], []

for run_name, split_map in protocols:
    print("\n" + "=" * 90)
    print("RUN:", run_name, {k: len(v) for k, v in split_map.items()})
    print("=" * 90)
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]

    deep_model, scalers, deep_result = train_advanced_model(run_name, split_map)
    for row in deep_result["history"]:
        all_history.append(row)
    row = metrics_row(run_name, "advanced_domain_robust_ser", "test", deep_result["test"], len(test_idx))
    row.update({"best_epoch": deep_result["best_epoch"], "best_val_macro_f1": deep_result["best_val_macro_f1"], "train_time_sec": deep_result["train_time_sec"]})
    all_metrics.append(row)

    svm_model, svm_result = fit_stats_svm(split_map)
    row = metrics_row(run_name, "rich_stats_rbf_svm", "test", svm_result["test"], len(test_idx))
    row.update({"best_epoch": 0, "best_val_macro_f1": svm_result["val"]["macro_f1"], "train_time_sec": svm_result["train_time_sec"]})
    all_metrics.append(row)

    X_meta_val = np.concatenate([deep_result["val"]["prob"], svm_result["val"]["prob"]], axis=1)
    X_meta_test = np.concatenate([deep_result["test"]["prob"], svm_result["test"]["prob"]], axis=1)
    meta = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    meta_start = time.time()
    meta.fit(X_meta_val, y[val_idx])
    meta_train_time = time.time() - meta_start
    t0 = time.time()
    prob = meta.predict_proba(X_meta_test)
    elapsed = time.time() - t0
    pred = prob.argmax(axis=1)
    yy = y[test_idx]
    stack_result = {
        "accuracy": accuracy_score(yy, pred),
        "macro_f1": f1_score(yy, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(yy, pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(yy, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(yy, pred, average="macro", zero_division=0),
        "y_true": yy,
        "y_pred": pred,
        "prob": prob,
        "elapsed_sec": elapsed,
    }
    row = metrics_row(run_name, "stacking_deep_plus_svm", "test", stack_result, len(test_idx))
    row.update({"best_epoch": 0, "best_val_macro_f1": np.nan, "train_time_sec": meta_train_time})
    all_metrics.append(row)

    pred_df = pd.DataFrame({
        "index": test_idx,
        "sample_id": metadata.loc[test_idx, "sample_id"].to_numpy(),
        "dataset": metadata.loc[test_idx, "dataset"].to_numpy(),
        "speaker_id": metadata.loc[test_idx, "speaker_id"].to_numpy(),
        "true_label": [ID_TO_LABEL[int(v)] for v in yy],
        "pred_label": [ID_TO_LABEL[int(v)] for v in pred],
    })
    for j, label in ID_TO_LABEL.items():
        pred_df[f"prob_{label}"] = prob[:, j]
    pred_path = PRED_DIR / f"predictions_{run_name}_stacking_deep_plus_svm.csv"
    pred_df.to_csv(pred_path, index=False)
    all_predictions.append(pred_df.assign(run_name=run_name))

metrics_df = pd.DataFrame(all_metrics)
history_df = pd.DataFrame(all_history)
metrics_path = REPORT_DIR / "advanced_model_metrics.csv"
history_path = REPORT_DIR / "advanced_model_history.csv"
metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)
display(metrics_df.sort_values(["run_name", "macro_f1"], ascending=[True, False]))
print("Saved:", metrics_path)
print("Saved:", history_path)


## 15. Visualizations and Diagnostics

Quan trọng khi đọc kết quả:

- Nếu `domain_accuracy` rất cao, fused embedding vẫn chứa dấu vết dataset rõ.
- Nếu bật adversarial mà emotion F1 giảm mạnh, `ADV_LAMBDA_MAX` quá lớn hoặc warm-up quá ngắn.
- Nếu `combined_random` vẫn thấp, kiểm tra riêng CREMA-D và lớp `fear/sad`.


In [ ]:
if len(metrics_df):
    plt.figure(figsize=(12, 6))
    sns.barplot(data=metrics_df, x="run_name", y="macro_f1", hue="model")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.title("Advanced Model Macro-F1 by Protocol")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "advanced_model_macro_f1.png", dpi=180)
    plt.show()

    best_df = metrics_df.loc[metrics_df.groupby("run_name")["macro_f1"].idxmax()].copy()
    display(best_df[["run_name", "model", "accuracy", "macro_f1", "inference_ms_per_sample"]])

    for _, row in best_df.iterrows():
        run_name = row["run_name"]
        pred_path = PRED_DIR / f"predictions_{run_name}_stacking_deep_plus_svm.csv"
        if not pred_path.exists():
            continue
        pdf = pd.read_csv(pred_path)
        cm = confusion_matrix(pdf["true_label"], pdf["pred_label"], labels=COMMON_EMOTIONS)
        plt.figure(figsize=(7, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS)
        plt.title(f"Confusion Matrix - {run_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"confusion_matrix_{run_name}.png", dpi=180)
        plt.show()


## 16. Reference Comparison Table

Các paper reference thường báo kết quả single-dataset/random split, không chắc speaker-aware. Vì vậy notebook lưu bảng này như **reported reference**, không coi là so sánh tuyệt đối công bằng.


In [ ]:
reference_rows = [
    {
        "model": "Ahmed et al. weighted ensemble 1D-CNN + CNN-LSTM + CNN-GRU",
        "protocol": "single-dataset, split not clearly strict speaker-aware",
        "reported_accuracy_text": "TESS 99.46%; RAVDESS 95.62%; SAVEE 93.22%; CREMA-D 90.47%",
        "main_idea": "Handcrafted features, augmentation, recurrent CNN variants, weighted ensemble.",
        "link": "https://arxiv.org/abs/2112.05666",
    },
    {
        "model": "Chowdhury et al. DCRF-BiLSTM feature engineering",
        "protocol": "individual and combined datasets, not clearly same as our strict split",
        "reported_accuracy_text": "RAVDESS 97.83%; SAVEE 97.02%; CREMA-D 95.10%; TESS 100%; R+T+S+E+C 93.76%",
        "main_idea": "Richer handcrafted acoustic features, augmentation, BiLSTM temporal modeling, DeepCRF.",
        "link": "https://arxiv.org/abs/2507.07046",
    },
    {
        "model": "Ullah et al. 1D-CNN feature fusion",
        "protocol": "combined 4-dataset, split details not fully reproducible from local materials",
        "reported_accuracy_text": "CREMA-D + RAVDESS + SAVEE + TESS: 92.62%",
        "main_idea": "ZCR + energy + entropy of energy + RMS + MFCC -> 1D-CNN.",
        "link": "https://doi.org/10.1109/ICIT56493.2022.9989197",
    },
    {
        "model": "DANN / Gradient Reversal Layer",
        "protocol": "domain adaptation method, not SER-specific accuracy",
        "reported_accuracy_text": "Used as regularization method, not direct SER benchmark.",
        "main_idea": "Learn task-discriminative but domain-invariant representation.",
        "link": "https://arxiv.org/abs/1505.07818",
    },
]
ref_df = pd.DataFrame(reference_rows)
ref_df.to_csv(REPORT_DIR / "advanced_reference_table.csv", index=False)
display(ref_df)


## 17. Save Output Package


In [ ]:
summary = {
    "notebook": "06_Advanced_Multi_Representation_Domain_Robust_SER",
    "architecture": "Temporal CNN-BiLSTM + Spectral 2D-CNN-SE + frozen WavLM adapter + stats branch + RBF-SVM + stacking + optional DANN/GRL",
    "output_dir": str(OUTPUT_DIR),
    "metrics_csv": str(REPORT_DIR / "advanced_model_metrics.csv"),
    "history_csv": str(REPORT_DIR / "advanced_model_history.csv"),
    "pretrained_speech_model": PRETRAINED_SPEECH_MODEL if X_speech_embed is not None else None,
    "use_domain_adversarial": USE_DOMAIN_ADVERSARIAL,
    "adv_lambda_max": ADV_LAMBDA_MAX,
    "notes": "Use ablation: no adversarial vs domain adversarial. Validation/test never use augmented variants.",
}
with open(REPORT_DIR / "advanced_model_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

zip_base = PROJECT_ROOT / "06_Advanced_Model_outputs_package"
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)
print("OUTPUT ZIP:", zip_path)
zip_path
